# One effect per perturbation combination

Fits, for each of the 1,041 response genes, a model in which **every distinct
perturbation combination is its own covariate** — the six single modules and
each observed module pair — with no interaction terms:

```
y ~ K_0 + ... + K_5 + K_0WK_1 + ... + K_4WK_5
```

Control cells are the reference level, so each coefficient is that
combination's effect relative to control. Because a pair has its own
coefficient rather than a product term, this model says how far a combination
moves a gene, not whether it does so additively.

Writes `ComboEffects_lm_residuals.rds`, read by Figures 5A, 5B, 5F and 5G.

## Setup

In [ ]:
%load_ext rpy2.ipython

import scanpy as sc
import pandas as pd
import anndata2ri
from rpy2.robjects import numpy2ri, pandas2ri

# The %%R cells below receive pandas frames; these converters are what the
# libraries.py star-import used to activate.
numpy2ri.activate()
pandas2ri.activate()
anndata2ri.activate()

DATASET = "/home/eraslab1/Projects/E3Ligase/analysisSingle/Notebooks/CombinatorialPerturbations/dataset"
MODULES = ["K_0", "K_1", "K_2", "K_3", "K_4", "K_5"]

OUT_FILE = "outputs/ComboEffects_lm_residuals.rds"

## Design: one column per combination

Each cell's perturbation indicators are collapsed into a single label — `K_0`,
`K_0WK_1`, `K_CONTROL` — and one-hot encoded. Cells carrying three or more
perturbations are dropped, leaving singles and doubles.

In [ ]:
adataSingles = sc.read(f"{DATASET}/adataTrainSingles.h5ad")
adataDoubles = sc.read(f"{DATASET}/adataDoubles.h5ad")
adata = sc.AnnData.concatenate(adataSingles, adataDoubles)

ALL_TERMS = MODULES + ["K_CONTROL"]

labels = adata.obs[ALL_TERMS].copy()
labels[labels == 0] = " "
for col in ALL_TERMS:
    labels.loc[labels[col] == 1, col] = col

combination = labels.apply(lambda x: "W".join(x[x != " "]), axis=1)
combination = combination[[len(x.split("W")) < 3 for x in combination]]   # singles and doubles

guideMatrix = pd.get_dummies(pd.DataFrame(combination), prefix="", prefix_sep="")
adata = adata[guideMatrix.index, :]

expressionMatrix = pd.DataFrame(adata.layers["ClusterResiduals"])
expressionMatrix.columns = adata.var_names
expressionMatrix.index = adata.obs.index

guideMatrix = guideMatrix.drop("K_CONTROL", axis=1)   # control is the reference level
allResp = adata.var_names

my_formula = "y~" + "+".join(guideMatrix.columns)

print(adata.shape, "|", guideMatrix.shape[1], "combinations")
print(my_formula)

## One model per gene

:::{note}
The original looped `seq(1, 1042, 1)` over 1,041 genes, so the last iteration
always failed and was swallowed by `tryCatch`. Looping over the actual column
count drops that silent error and leaves the result unchanged.
:::

In [ ]:
%%R -i guideMatrix,expressionMatrix,my_formula,allResp,OUT_FILE
library(broom)

coefDF <- data.frame()

for (i in seq_len(ncol(expressionMatrix))) {
    tryCatch({
        guideMatrix["y"] <- expressionMatrix[, i]
        myFit <- lm(formula(my_formula), data = guideMatrix)
        myDF  <- data.frame(tidy(myFit))
        myDF$respGene <- allResp[i]
        coefDF <- rbind(coefDF, myDF)
    }, error = function(e) message("gene ", i, " skipped: ", conditionMessage(e)))
}

saveRDS(coefDF, OUT_FILE)
dim(coefDF)